In [1]:
"""
DATA PREPARATION
- Load raw data
- Deduplicate (one row per player per season per team)
- Clean decimals, positions, leagues
- Engineer features
- Save cleaned file
"""
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

def load_and_clean(path="data/Top5_League_Players_2017to2024_dataset.csv"):

    # ── Load ──────────────────────────────────────────
    df = pd.read_csv(path, sep=';', on_bad_lines='skip', decimal=',')
    print(f"Raw shape: {df.shape}")

    # ── Deduplicate ───────────────────────────────────
    before = len(df)
    df = df.drop_duplicates(subset=['player', 'season', 'team'])
    print(f"Removed {before - len(df)} duplicate rows → {len(df)} rows remaining")

    # ── Clean league names ────────────────────────────
    league_map = {
        'ENG-Premier League': 'Premier League',
        'ESP-La Liga':        'La Liga',
        'FRA-Ligue 1':        'Ligue 1',
        'GER-Bundesliga':     'Bundesliga',
        'ITA-Serie A':        'Serie A',
    }
    df['league'] = df['league'].map(league_map).fillna(df['league'])

    # ── Season labels ─────────────────────────────────
    season_map = {
        1718:'2017-18', 1819:'2018-19', 1920:'2019-20',
        2021:'2020-21', 2122:'2021-22', 2223:'2022-23',
        2324:'2023-24', 2425:'2024-25',
    }
    df['season_label'] = df['season'].map(season_map)

    # ── Primary position ──────────────────────────────
    def primary_pos(p):
        if pd.isna(p): return 'Unknown'
        pos = str(p).split(',')[0].strip()
        return pos if pos in ['GK','DF','MF','FW'] else 'Unknown'
    df['primary_pos'] = df['pos_'].apply(primary_pos)

    # ── Filter min 5 matches ──────────────────────────
    df = df[df['Playing Time_MP'] >= 5].reset_index(drop=True)

    # ── Fill nulls ────────────────────────────────────
    num_cols = df.select_dtypes(include=np.number).columns
    df[num_cols] = df[num_cols].fillna(0)

    # ── Feature engineering ───────────────────────────
    mins90 = df['90s_'].replace(0, np.nan)
    df['goal_contrib_p90']      = (df['Performance_Gls'] + df['Performance_Ast']) / mins90
    df['xG_overperformance']    = df['Performance_Gls'] - df['Expected_xG']
    df['progressive_impact_p90']= (df['Progression_PrgC'] + df['Progression_PrgP']) / mins90
    df['defensive_actions_p90'] = (df['Tackles_TklW'] + df['Performance_Int'] + df['Blocks_Blocks']) / mins90
    total_aerials = df['Aerial Duels_Won'] + df['Aerial Duels_Lost']
    df['aerial_dominance']      = df['Aerial Duels_Won'] / total_aerials.replace(0, np.nan)
    df['shot_conversion']       = df['Performance_Gls'] / df['Standard_Sh'].replace(0, np.nan)
    df[['goal_contrib_p90','progressive_impact_p90','defensive_actions_p90',
        'aerial_dominance','shot_conversion']] = \
        df[['goal_contrib_p90','progressive_impact_p90','defensive_actions_p90',
            'aerial_dominance','shot_conversion']].fillna(0)

    print(f"Final shape: {df.shape}")
    return df

if __name__ == '__main__':
    df = load_and_clean()
    df.to_csv('data/cleaned.csv', index=False)
    print("Saved → data/cleaned.csv")


Raw shape: (22929, 178)
Removed 511 duplicate rows → 22418 rows remaining
Final shape: (18616, 186)
Saved → data/cleaned.csv


In [2]:
"""
FORWARD ROLE CLASSIFICATION
Step 1 — KMeans clustering to discover roles (unsupervised)
Step 2 — Random Forest classifier trained on those labels
Step 3 — Evaluation: confusion matrix + feature importance saved as images
Roles: Poacher | Winger | Target Man | Pressing Forward
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os, warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix

os.makedirs('models', exist_ok=True)
os.makedirs('plots', exist_ok=True)

plt.rcParams.update({
    "figure.facecolor": "#0d1117", "axes.facecolor": "#161b22",
    "axes.edgecolor":   "#30363d", "axes.labelcolor": "#e6edf3",
    "xtick.color":      "#8b949e", "ytick.color":     "#8b949e",
    "text.color":       "#e6edf3", "grid.color":      "#21262d",
    "font.family":      "DejaVu Sans",
})

df = pd.read_csv('data/cleaned.csv')

# ── Filter forwards with enough playing time ──────────
fwd = df[
    (df['primary_pos'] == 'FW') &
    (df['90s_'] >= 8)
].copy()
print(f"Forwards with >=8 90s: {len(fwd)}")

# ── Features ──────────────────────────────────────────
role_features = [
    'shot_conversion',
    'Touches_Att Pen',
    'Take-Ons_Succ',
    'Progression_PrgC',
    'Pass Types_Crs',
    'aerial_dominance',
    'Aerial Duels_Won',
    'Expected_xG',
    'defensive_actions_p90',
    'Tackles_TklW',
    'Performance_Fls',
    'goal_contrib_p90',
    'SCA_SCA',
]
role_features = [f for f in role_features if f in fwd.columns]

# ── Aggregate per player — career average ─────────────
fwd_agg = fwd.groupby('player')[role_features + ['league', 'age_']].agg(
    {**{f: 'mean' for f in role_features}, 'league': 'last', 'age_': 'mean'}
).reset_index()

X = fwd_agg[role_features].fillna(0)

# ── Normalise ─────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ── KMeans — 4 clusters ───────────────────────────────
kmeans = KMeans(n_clusters=4, random_state=42, n_init=20)
fwd_agg['cluster'] = kmeans.fit_predict(X_scaled)

# ── Interpret clusters ────────────────────────────────
centers = pd.DataFrame(
    scaler.inverse_transform(kmeans.cluster_centers_),
    columns=role_features
)
print("\nCluster Centers (key stats):")
key_stats = ['shot_conversion', 'aerial_dominance', 'Take-Ons_Succ',
             'defensive_actions_p90', 'Touches_Att Pen']
key_stats = [k for k in key_stats if k in centers.columns]
print(centers[key_stats].round(3))

role_mapping = {}
role_mapping[centers['shot_conversion'].idxmax()]       = 'Poacher'
role_mapping[centers['Take-Ons_Succ'].idxmax()]         = 'Winger'
role_mapping[centers['aerial_dominance'].idxmax()]      = 'Target Man'
role_mapping[centers['defensive_actions_p90'].idxmax()] = 'Pressing Forward'

all_roles = ['Poacher', 'Winger', 'Target Man', 'Pressing Forward']
used = set(role_mapping.values())
remaining = [r for r in all_roles if r not in used]
for c in range(4):
    if c not in role_mapping:
        role_mapping[c] = remaining.pop(0) if remaining else 'Winger'

fwd_agg['role'] = fwd_agg['cluster'].map(role_mapping)
print("\nRole Distribution:")
print(fwd_agg['role'].value_counts())

# ── Map labels back to season rows ────────────────────
player_role_map = fwd_agg.set_index('player')['role'].to_dict()
fwd['role'] = fwd['player'].map(player_role_map)
fwd = fwd.dropna(subset=['role'])

# ── Random Forest Classifier ──────────────────────────
X_clf = fwd[role_features].fillna(0)
y_clf = fwd['role']

X_tr, X_te, y_tr, y_te = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

clf = RandomForestClassifier(
    n_estimators=200, max_depth=10, random_state=42, n_jobs=-1
)
clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)

cv = cross_val_score(clf, X_clf, y_clf, cv=5, scoring='accuracy')
print(f"\nCross-Val Accuracy: {cv.mean()*100:.1f}% +/- {cv.std()*100:.1f}%")
print(f"Test Accuracy:      {(y_pred==y_te).mean()*100:.1f}%")
print("\nClassification Report:")
print(classification_report(y_te, y_pred))

# ── PLOT 1: Confusion Matrix ──────────────────────────
print("Saving confusion matrix...")
cm     = confusion_matrix(y_te, y_pred, labels=all_roles)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=all_roles, yticklabels=all_roles,
    linewidths=0.5, linecolor='#21262d', ax=ax,
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Forward Role Classifier — Confusion Matrix\nRandom Forest (200 trees, 5-fold CV: 75%)',
             fontsize=13, fontweight='bold', pad=14)
ax.set_xlabel('Predicted Role', labelpad=10)
ax.set_ylabel('Actual Role',    labelpad=10)
plt.tight_layout()
plt.savefig('plots/role_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved -> plots/role_confusion_matrix.png")

# ── PLOT 2: Feature Importance ────────────────────────
print("Saving feature importance...")
feature_labels = {
    'shot_conversion':        'Shot Conversion',
    'Touches_Att Pen':        'Penalty Area Touches',
    'Take-Ons_Succ':          'Successful Dribbles',
    'Progression_PrgC':       'Progressive Carries',
    'Pass Types_Crs':         'Crosses',
    'aerial_dominance':       'Aerial Win Rate',
    'Aerial Duels_Won':       'Aerial Duels Won',
    'Expected_xG':            'Expected Goals (xG)',
    'defensive_actions_p90':  'Defensive Actions/90',
    'Tackles_TklW':           'Tackles Won',
    'Performance_Fls':        'Fouls Committed',
    'goal_contrib_p90':       'Goal Contribution/90',
    'SCA_SCA':                'Shot Creating Actions',
}
importances = pd.Series(
    clf.feature_importances_,
    index=[feature_labels.get(f, f) for f in role_features]
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
colors  = plt.cm.YlOrRd(np.linspace(0.35, 0.9, len(importances)))
ax.barh(importances.index, importances.values,
        color=colors, edgecolor='none', height=0.65)
for i, (val, name) in enumerate(zip(importances.values, importances.index)):
    ax.text(val + 0.002, i, f'{val:.3f}', va='center', fontsize=9)
ax.set_title('Forward Role Classifier — Feature Importance\nRandom Forest',
             fontsize=13, fontweight='bold', pad=14)
ax.set_xlabel('Importance Score', labelpad=10)
ax.grid(True, axis='x', alpha=0.3)
ax.set_xlim(0, importances.max() + 0.05)
plt.tight_layout()
plt.savefig('plots/role_feature_importance.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved -> plots/role_feature_importance.png")

# ── Save model ────────────────────────────────────────
joblib.dump({
    'kmeans':       kmeans,
    'scaler':       scaler,
    'classifier':   clf,
    'features':     role_features,
    'role_mapping': role_mapping,
    'cv_accuracy':  round(cv.mean() * 100, 1),
    'test_accuracy': round((y_pred == y_te).mean() * 100, 1),
}, 'models/forward_roles.pkl')

fwd_agg.to_csv('data/forward_roles.csv', index=False)
print("\nSaved -> models/forward_roles.pkl")
print("Saved -> data/forward_roles.csv")
print("\nDONE")


Forwards with >=8 90s: 3119

Cluster Centers (key stats):
   shot_conversion  aerial_dominance  Take-Ons_Succ  defensive_actions_p90  \
0            0.130             0.314         47.862                  2.101   
1            0.160             0.443         18.728                  1.369   
2            0.080             0.329         24.677                  2.720   
3            0.134             0.376         12.477                  1.469   

   Touches_Att Pen  
0          105.749  
1          103.108  
2           49.631  
3           51.848  

Role Distribution:
role
Poacher             412
Pressing Forward    337
Target Man          257
Winger              211
Name: count, dtype: int64

Cross-Val Accuracy: 75.0% +/- 2.1%
Test Accuracy:      76.6%

Classification Report:
                  precision    recall  f1-score   support

         Poacher       0.70      0.75      0.72       172
Pressing Forward       0.80      0.72      0.76       120
      Target Man       0.76      0.74 

In [3]:

import pandas as pd
import numpy as np
import joblib, os, warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.metrics.pairwise import cosine_similarity

os.makedirs('models', exist_ok=True)
df = pd.read_csv('data/cleaned.csv')

# ── Features ──────────────────────────────────────────
sim_features = [
    # Attacking output
    'goal_contrib_p90',        # goals + assists per 90
    'shot_conversion',         # goals per shot — clinical finishing
    'Expected_xG',             # quality of chances for themselves
    'Expected_xAG',            # quality of chances created for teammates
    'Standard_Dist',           # average shot distance — box striker vs long range

    # Chance creation style
    'SCA_SCA',                 # shot creating actions
    'GCA_GCA',                 # goal creating actions
    'KP_',                     # key passes
    'PPA_',                    # passes into penalty area — direct danger
    'CrsPA_',                  # crosses into penalty area — crossing style
    '1/3_',                    # passes into final third — build-up involvement

    # Progression
    'progressive_impact_p90',  # combined progressive carries + passes per 90
    'Touches_Att Pen',         # penalty area presence

    # Dribbling
    'Take-Ons_Att',            # dribble attempts — how often they try
    'Take-Ons_Succ%',          # dribble success rate — quality of attempts

    # Defensive contribution
    'defensive_actions_p90',   # combined tackles + interceptions + blocks per 90
    'aerial_dominance',        # aerial duel win rate
    'Performance_Int',         # interceptions — reading of the game
    'Tackles_TklW',            # tackles won — active defending
    'Performance_Fls',         # fouls committed — pressing aggression

    # Ball retention
    'Carries_Dis',             # times dispossessed — ball retention under pressure
]
sim_features = [f for f in sim_features if f in df.columns]
print(f"Total features: {len(sim_features)}")

# ── Aggregate per player — career average ─────────────
feat_agg = df.groupby('player')[sim_features].mean()
meta_agg = df.groupby('player').agg(
    primary_pos=('primary_pos',      'last'),
    league=('league',                'last'),
    team=('team',                    'last'),
    avg_age=('age_',                 'mean'),
    career_goals=('Performance_Gls', 'sum'),
    career_assists=('Performance_Ast','sum'),
    seasons=('season_label',         'nunique'),
).reset_index()

player_agg = meta_agg.merge(feat_agg.reset_index(), on='player')
X = player_agg[sim_features].fillna(0)

# ── Step 1: Standardise ───────────────────────────────
# Mean 0, std 1 — puts all features on equal scale
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ── Step 2: Unit normalise ────────────────────────────
# Normalise each player vector to length 1
# This makes Euclidean distance = cosine distance
# So KMeans clusters by style direction not volume
X_unit = normalize(X_scaled, norm='l2')
print("\nVector lengths after unit normalisation (should all be 1.0):")
lengths = np.linalg.norm(X_unit, axis=1)
print(f"  Min: {lengths.min():.4f}  Max: {lengths.max():.4f}  Mean: {lengths.mean():.4f}")

# ── Step 3: KMeans with 15 clusters ───────────────────
# 15 clusters aligns with real football structure:
#   GK: ~2 styles, DF: ~4, MF: ~4, FW: ~4 = 14 to 16
print("\nRunning KMeans (15 clusters) on unit-normalised vectors...")
kmeans = KMeans(n_clusters=15, random_state=42, n_init=20)
player_agg['style_cluster'] = kmeans.fit_predict(X_unit)

# ── Cluster summary ───────────────────────────────────
print("\nCluster sizes and position mix:")
for cluster in range(15):
    cluster_players = player_agg[player_agg['style_cluster'] == cluster]
    pos_mix = cluster_players['primary_pos'].value_counts().head(3).to_dict()
    print(f"  Cluster {cluster:2d} ({len(cluster_players):3d} players): {pos_mix}")

# ── Step 4: Cosine similarity within each cluster ─────
print("\nComputing cosine similarity within clusters...")
cluster_sim_matrices = {}
for cluster in range(15):
    mask            = player_agg['style_cluster'] == cluster
    cluster_players = player_agg[mask]
    if len(cluster_players) < 2:
        continue
    cluster_X = X_unit[mask]
    sim_mat   = cosine_similarity(cluster_X)
    cluster_sim_matrices[cluster] = pd.DataFrame(
        sim_mat,
        index=cluster_players['player'].values,
        columns=cluster_players['player'].values
    )
print(f"  Built {len(cluster_sim_matrices)} cluster similarity matrices")

# ── Save ──────────────────────────────────────────────
joblib.dump({
    'kmeans':               kmeans,
    'scaler':               scaler,
    'player_agg':           player_agg,
    'cluster_sim_matrices': cluster_sim_matrices,
    'features':             sim_features,
    'n_clusters':           15,
}, 'models/player_similarity.pkl')
print("Saved -> models/player_similarity.pkl")

# ── Helper function ───────────────────────────────────
def find_similar(name, top_n=5, same_pos=True, max_age=None):
    if name not in player_agg['player'].values:
        return f"Player not found: {name}"
    player_row = player_agg[player_agg['player'] == name].iloc[0]
    cluster_id = int(player_row['style_cluster'])
    if cluster_id not in cluster_sim_matrices:
        return "No similarity data for this cluster"
    sim_mat = cluster_sim_matrices[cluster_id]
    if name not in sim_mat.index:
        return "Player not in similarity matrix"
    scores = sim_mat[name].drop(name).sort_values(ascending=False)
    result = player_agg[player_agg['player'].isin(scores.index)].copy()
    result['similarity'] = result['player'].map(scores)
    if same_pos:
        result = result[result['primary_pos'] == player_row['primary_pos']]
    if max_age and max_age > 0:
        result = result[result['avg_age'] <= max_age]
    return result.sort_values('similarity', ascending=False).head(top_n)[
        ['player','league','team','primary_pos','avg_age','career_goals','similarity']
    ]

# ── Test ──────────────────────────────────────────────
print("\nSimilar to Kevin De Bruyne:")
print(find_similar('Kevin De Bruyne').to_string(index=False))

print("\nSimilar to Lionel Messi (under 28):")
print(find_similar('Lionel Messi', max_age=28).to_string(index=False))

print("\nSimilar to Virgil van Dijk:")
print(find_similar('Virgil van Dijk').to_string(index=False))

print("\nSimilar to Alisson:")
print(find_similar('Alisson').to_string(index=False))

print("\nSimilar to Erling Haaland:")
print(find_similar('Erling Haaland').to_string(index=False))


Total features: 21

Vector lengths after unit normalisation (should all be 1.0):
  Min: 1.0000  Max: 1.0000  Mean: 1.0000

Running KMeans (15 clusters) on unit-normalised vectors...

Cluster sizes and position mix:
  Cluster  0 (381 players): {'DF': 248, 'MF': 105, 'FW': 27}
  Cluster  1 (603 players): {'FW': 297, 'MF': 265, 'DF': 41}
  Cluster  2 (325 players): {'FW': 295, 'MF': 30}
  Cluster  3 (381 players): {'DF': 259, 'MF': 105, 'FW': 16}
  Cluster  4 (411 players): {'MF': 248, 'DF': 82, 'FW': 81}
  Cluster  5 (387 players): {'GK': 349, 'DF': 20, 'FW': 9}
  Cluster  6 (372 players): {'MF': 309, 'DF': 60, 'FW': 3}
  Cluster  7 (317 players): {'FW': 209, 'MF': 62, 'GK': 31}
  Cluster  8 (530 players): {'DF': 500, 'MF': 30}
  Cluster  9 (276 players): {'MF': 144, 'FW': 104, 'DF': 28}
  Cluster 10 (409 players): {'DF': 355, 'MF': 39, 'FW': 15}
  Cluster 11 (367 players): {'FW': 334, 'MF': 31, 'DF': 2}
  Cluster 12 (321 players): {'MF': 165, 'DF': 149, 'FW': 7}
  Cluster 13 (435 player